# Pipeline NLP para Clasificación de Textos Históricos por Década

**Estrategia central:** mantener **dos vistas** del texto —una conservadora (text_base) que preserva la huella temporal original y otra normalizada (text_norm_soft) para robustez del modelo— y nunca eliminar señal histórica bajo la apariencia de "limpieza".


## Configuración y dependencias


In [ ]:
import pandas as pd
import numpy as np
import re
import json
import warnings
from collections import Counter
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import nltk
from nltk.corpus import stopwords

warnings.filterwarnings('ignore')
np.random.seed(42)

for res in ['tokenizers/punkt', 'corpora/stopwords']:
    try:
        nltk.data.find(res)
    except LookupError:
        nltk.download(res.split('/')[0], quiet=True)

RUTA_DATA = Path("../data")
RUTA_OUT = Path("../artifacts")
RUTA_OUT.mkdir(exist_ok=True)

print("Configuración lista.")


## Carga de datos


In [ ]:
df_train = pd.read_csv(RUTA_DATA / "train.csv")
df_eval = pd.read_csv(RUTA_DATA / "eval.csv")
df_train["decade"] = df_train["decade"].astype(int)
print(f"Train: {df_train.shape}, Eval: {df_eval.shape}")
print(f"Décadas: {sorted(df_train['decade'].unique())}")
print(f"Distribución:\n{df_train['decade'].value_counts().sort_index()}")


---
## 1. Limpieza y normalización


### Paso 1. Ingesta y validación UTF-8
**Objetivo:** asegurar codificación consistente y detectar caracteres rotos.

**Output:** `raw_text`, `text_utf8`, `encoding_source`, `decode_flag`


In [ ]:
def validate_utf8_series(series):
    records = []
    for raw in series:
        if not isinstance(raw, str):
            s = str(raw) if raw is not None else ""
            records.append({"raw_text": s, "text_utf8": s,
                            "encoding_source": "coerced", "decode_flag": 0})
            continue
        try:
            raw.encode("utf-8").decode("utf-8")
            records.append({"raw_text": raw, "text_utf8": raw,
                            "encoding_source": "utf-8", "decode_flag": 1})
        except (UnicodeDecodeError, UnicodeEncodeError):
            try:
                text = raw.encode("latin1").decode("utf-8", errors="replace")
                records.append({"raw_text": raw, "text_utf8": text,
                                "encoding_source": "latin1->utf-8", "decode_flag": 0})
            except Exception:
                text = raw.encode("utf-8", errors="replace").decode("utf-8")
                records.append({"raw_text": raw, "text_utf8": text,
                                "encoding_source": "forced_replace", "decode_flag": 0})
    return pd.DataFrame(records)

df_utf8 = validate_utf8_series(df_train["text"])
df_train = pd.concat([df_utf8, df_train.drop(columns=["text"])], axis=1)

print(f"decode_flag distribution:\n{df_train['decode_flag'].value_counts()}")
print(f"encoding_source distribution:\n{df_train['encoding_source'].value_counts()}")
print(f"UTF-8 OK: {df_train['decode_flag'].sum()} / {len(df_train)}")


### Paso 2. Crear doble representación del texto
**Objetivo:** no perder rasgos paleográficos al limpiar.

**Output:** `text_raw`, `text_base`, `text_norm_soft`


In [ ]:
def normalize_whitespace_soft(text):
    text = re.sub(r'[ \\t]+', ' ', text)
    text = re.sub(r'\\n{3,}', '\\n\\n', text)
    return text.strip()

df_train["text_raw"] = df_train["raw_text"]
df_train["text_base"] = df_train["text_utf8"]
df_train["text_norm_soft"] = df_train["text_utf8"].apply(normalize_whitespace_soft)

diffs = (df_train["text_base"] != df_train["text_norm_soft"]).sum()
print(f"Muestras con diferencias tras normalización suave: {diffs} / {len(df_train)}")


### Paso 3. Normalizar controles invisibles y espacios
**Objetivo:** eliminar ruido técnico no lingüístico sin borrar ruido histórico real.

**Output:** texto sin caracteres de control, con espaciado estable.


In [ ]:
def strip_control_chars(text):
    chars = []
    for ch in text:
        cp = ord(ch)
        if ch in ("\\n", "\\t"):
            chars.append(ch)
        elif cp < 0x20 or cp == 0x7F:
            continue
        else:
            chars.append(ch)
    return "".join(chars)

df_train["text_base"] = df_train["text_base"].apply(strip_control_chars)
df_train["text_norm_soft"] = df_train["text_norm_soft"].apply(strip_control_chars)
df_train["text_norm_soft"] = df_train["text_norm_soft"].str.replace("\\t", " ")
df_train["text_norm_soft"] = df_train["text_norm_soft"].apply(
    lambda t: re.sub(r' +', ' ', t))

chars_before = sum(len(t) for t in df_train["text_utf8"])
chars_after = sum(len(t) for t in df_train["text_base"])
print(f"Caracteres antes: {chars_before:,} -> después: {chars_after:,} "
      f"(eliminados {chars_before - chars_after:,})")


### Paso 4. Resolver saltos de línea con guion de corte
**Objetivo:** reconstruir palabras partidas por composición tipográfica. Marcar el fenómeno como señal temporal.

**Output:** `text_dehyphenated`, `count_linebreak_hyphen`, `had_linebreak_hyphen`


In [ ]:
HYPHEN_PATTERN = re.compile(
    r'([a-zA-Z\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc\u00f1\u00c1\u00c9\u00cd\u00d3\u00da\u00dc\u00d1\u017ff]{2,})-\\n([a-zA-Z\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc\u00f1\u00c1\u00c9\u00cd\u00d3\u00da\u00dc\u00d1]{2,})'
)

def resolve_linebreak_hyphens(text):
    matches = list(HYPHEN_PATTERN.finditer(text))
    count = len(matches)
    dehyphenated = HYPHEN_PATTERN.sub(r'\\1\\2', text)
    return dehyphenated, count

results = df_train["text_base"].apply(resolve_linebreak_hyphens)
df_train["text_dehyphenated"] = [r[0] for r in results]
df_train["count_linebreak_hyphen"] = [r[1] for r in results]
df_train["had_linebreak_hyphen"] = (df_train["count_linebreak_hyphen"] > 0).astype(int)

print(f"Muestras con guion de corte: {df_train['had_linebreak_hyphen'].sum()} / {len(df_train)}")
print(f"Total guiones de corte resueltos: {df_train['count_linebreak_hyphen'].sum()}")
print(f"Stats: media={df_train['count_linebreak_hyphen'].mean():.2f}, max={df_train['count_linebreak_hyphen'].max()}")


### Paso 5. Separar ruido editorial de contenido
**Objetivo:** detectar citas bibliográficas, numeración marginal, referencias y metadatos incrustados.

**Output:** `text_main`, `editorial_spans`, `citation_flag`, `numeric_reference_density`


In [ ]:
PATTERN_CITATION_5 = re.compile(
    r'(?:lib|cap|pag|fol|epist|tom|vol|num|p\\u00e1g|col|par|p\\.|cap\\.|lib\\.|fol\\.|epist\\.|tom\\.|vol\\.)'
    r'\\.?\\s*\\d+(?:[.,]\\s*\\d+)*',
    re.IGNORECASE
)
PATTERN_MARGINAL_NUM = re.compile(r'^\\s*\\(?\\d+\\)?\\s+', re.MULTILINE)
PATTERN_DENSE_PUNCT = re.compile(r'(?:[.,;:]{4,})')

def detect_editorial_noise(text):
    citations = list(PATTERN_CITATION_5.finditer(text))
    marginal = list(PATTERN_MARGINAL_NUM.finditer(text))
    dense_punct = list(PATTERN_DENSE_PUNCT.finditer(text))
    spans = [(m.start(), m.end(), "citation") for m in citations]
    spans += [(m.start(), m.end(), "marginal_num") for m in marginal]
    spans += [(m.start(), m.end(), "dense_punct") for m in dense_punct]
    spans.sort()
    citation_count = len(citations)
    numeric_ref_count = sum(1 for m in marginal)
    total_chars = max(len(text), 1)
    numeric_ref_density = (numeric_ref_count * 100) / total_chars if total_chars else 0
    return {
        "editorial_spans": spans,
        "citation_count": citation_count,
        "citation_flag": int(citation_count > 0 or numeric_ref_count > 0),
        "numeric_reference_density": round(numeric_ref_density, 6),
    }

results_ed = df_train["text_base"].apply(detect_editorial_noise)
df_train["citation_count"] = [r["citation_count"] for r in results_ed]
df_train["citation_flag"] = [r["citation_flag"] for r in results_ed]
df_train["numeric_reference_density"] = [r["numeric_reference_density"] for r in results_ed]

print(f"Muestras con citas/referencias: {df_train['citation_flag'].sum()} / {len(df_train)}")
print(f"Media densidad refs numéricas: {df_train['numeric_reference_density'].mean():.6f}")


### Paso 6. Etiquetar ruido OCR en vez de eliminarlo ciegamente
**Objetivo:** tratar el OCR como señal útil de época/impreso y también como fuente de degradación.

**Output:** `ocr_noise_score`, `weird_char_rate`, `alnum_mixed_token_rate`, `suspect_tokens_list`


In [ ]:
SUSPECT_CHARS = set('^|~`@#$%^&*_={}[]\\\\|<>')
ALNUM_MIXED_PATTERN = re.compile(r'\\b(?=[a-zA-Z]*\\d)(?=\\d*[a-zA-Z])[a-zA-Z0-9]+\\b')
WEIRD_SEQUENCE_PATTERN = re.compile(r'[^\\w\\s\\u00e1\\u00e9\\u00ed\\u00f3\\u00fa\\u00fc\\u00f1\\u00c1\\u00c9\\u00cd\\u00d3\\u00da\\u00dc\\u00d1\\.,;:\\-\\"\\'\\u00ab\\u00bb\\(\\)\\n\\t]{2,}')

def score_ocr_noise(text):
    if not text or len(text) == 0:
        return {"ocr_noise_score": 1.0, "weird_char_rate": 1.0,
                "alnum_mixed_token_rate": 0.0, "suspect_tokens_list": []}
    total_chars = len(text)
    total_tokens = len(text.split())
    weird_chars = sum(1 for c in text if c in SUSPECT_CHARS)
    weird_char_rate = weird_chars / max(total_chars, 1)
    weird_seq_matches = list(WEIRD_SEQUENCE_PATTERN.findall(text))
    alnum_mixed = list(ALNUM_MIXED_PATTERN.findall(text))
    alnum_rate = len(alnum_mixed) / max(total_tokens, 1)
    suspect_tokens = [t for t in text.split() if
                      any(c in SUSPECT_CHARS for c in t) or
                      (len(t) > 1 and any(c.isdigit() for c in t) and any(c.isalpha() for c in t))]
    ocr_score = min(1.0, weird_char_rate * 10 + alnum_rate * 3 + len(weird_seq_matches) / max(total_tokens, 1) * 5)
    return {
        "ocr_noise_score": round(ocr_score, 6),
        "weird_char_rate": round(weird_char_rate, 6),
        "alnum_mixed_token_rate": round(alnum_rate, 6),
        "suspect_tokens_list": suspect_tokens[:20],
    }

results_ocr = df_train["text_base"].apply(score_ocr_noise)
df_train["ocr_noise_score"] = [r["ocr_noise_score"] for r in results_ocr]
df_train["weird_char_rate"] = [r["weird_char_rate"] for r in results_ocr]
df_train["alnum_mixed_token_rate"] = [r["alnum_mixed_token_rate"] for r in results_ocr]
df_train["suspect_tokens_list"] = [r["suspect_tokens_list"] for r in results_ocr]

print(f"OCR noise score stats: min={df_train['ocr_noise_score'].min():.4f}, mean={df_train['ocr_noise_score'].mean():.4f}, max={df_train['ocr_noise_score'].max():.4f}")
print(f"Top-5 ruidosas:")
print(df_train.nlargest(5, 'ocr_noise_score')[['ocr_noise_score', 'text_base']].to_string())


---
## 2. Ingeniería de características


### Paso 7. Extraer métricas de estilometría y puntuación
**Objetivo:** capturar la huella de maquetación y puntuación histórica.

**Output:** `semicolon_rate`, `emdash_rate`, `quote_angle_rate`, `sentence_len_mean`, `linebreak_hyphen_rate`


In [ ]:
def extract_stylometry(text):
    if not text:
        return {k: 0.0 for k in ["semicolon_rate", "emdash_rate", "colon_rate",
                "paren_rate", "quote_angle_rate", "excl_rate", "quest_rate",
                "comma_rate", "asterisk_rate", "dash_rate", "quote_double_rate",
                "sentence_len_mean"]}
    n = max(len(text), 1)
    words = text.split()
    nw = max(len(words), 1)
    sents = [s for s in re.split(r'[.!?]+', text) if s.strip()]
    sent_lens = [len(s.split()) for s in sents] if sents else [nw]
    return {
        "semicolon_rate": text.count(";") * 100 / n,
        "emdash_rate": (text.count("\u2014") + text.count("\u2013")) * 100 / n,
        "colon_rate": text.count(":") * 100 / n,
        "paren_rate": (text.count("(") + text.count(")")) * 100 / n,
        "quote_angle_rate": (text.count("\u00ab") + text.count("\u00bb")) * 100 / n,
        "excl_rate": text.count("!") * 100 / n,
        "quest_rate": text.count("?") * 100 / n,
        "comma_rate": text.count(",") * 100 / n,
        "asterisk_rate": text.count("*") * 100 / n,
        "dash_rate": text.count("-") * 100 / n,
        "quote_double_rate": text.count('"') * 100 / n,
        "sentence_len_mean": round(np.mean(sent_lens), 4),
    }

stylo_df = df_train["text_base"].apply(lambda t: pd.Series(extract_stylometry(t)))
df_train = pd.concat([df_train, stylo_df], axis=1)

print("Estilometría extraída. Columnas añadidas:")
print(list(stylo_df.columns))


### Paso 8. Extraer rasgos de ortografía histórica
**Objetivo:** medir evolución gráfica del español.

**Output:** `obsolete_accent_rate`, `long_s_proxy_rate`, `latin_abbrev_rate`, `honorific_caps_rate`


In [ ]:
LATIN_ABBREV_PATTERN = re.compile(
    r'\\b(?:lib|cap|pag|fol|epist|tom|vol|num|p\\u00e1g|col|par|ss|ib|id|op|cit|et|ca|vs)\\.',
    re.IGNORECASE
)
HONORIFIC_PATTERN = re.compile(
    r'\\b(?:D\\.|Don|Do\u00f1a|Exc\\.|S\\.M\\.|S\\.A\\.|R\\.P\\.|V\\.E\\.|V\\.S\\.|Fray|Sor|Sto|Sta)\\.?',
)
LONG_S_MARKERS = re.compile(r'\u017f')  # ſ

def extract_historical_spelling(text, decade=None):
    n = max(len(text), 1)
    nw = max(len(text.split()), 1)
    long_s_count = len(LONG_S_MARKERS.findall(text))
    obsolete_accent = sum(1 for c in text if c in '\u00c1\u00c9\u00cd\u00d3\u00da')
    latin_abbrevs = len(LATIN_ABBREV_PATTERN.findall(text))
    honorific_caps = len(HONORIFIC_PATTERN.findall(text))
    return {
        "obsolete_accent_rate": obsolete_accent * 100 / n,
        "long_s_proxy_rate": long_s_count * 100 / n,
        "latin_abbrev_rate": latin_abbrevs * 100 / nw,
        "honorific_caps_rate": honorific_caps * 100 / nw,
    }

hist_df = df_train["text_base"].apply(lambda t: pd.Series(extract_historical_spelling(t)))
df_train = pd.concat([df_train, hist_df], axis=1)

print(f"Muestras con \u017f (s larga): {(df_train['long_s_proxy_rate'] > 0).sum()}")
print(f"Muestras con abreviaturas latinas: {(df_train['latin_abbrev_rate'] > 0).sum()}")
print(f"Muestras con may\u00fasculas honor\u00edficas: {(df_train['honorific_caps_rate'] > 0).sum()}")


### Paso 9. Extraer rasgos de ruido OCR
**Objetivo:** convertir corrupción visual en variables modelables.

**Output:** `ocr_symbol_rate`, `oov_rate`, `corrupt_token_rate`


In [ ]:
all_words = \" \".join(df_train["text_base"]).lower().split()
word_freq = Counter(all_words)
historical_lexicon = set(w for w, c in word_freq.most_common(20000))

spanish_stops = set(stopwords.words("spanish"))
modern_lexicon = historical_lexicon | spanish_stops | {
    "el", "la", "los", "las", "de", "del", "en", "un", "una", "y", "e", "o",
    "a", "ante", "bajo", "con", "contra", "para", "por", "que", "su", "le",
    "se", "no", "es", "lo", "como", "m\u00e1s", "pero", "si", "este", "esta", "entre"
}

def extract_ocr_features(text):
    nw = max(len(text.split()), 1)
    tokens = text.split()
    non_alpha = sum(1 for t in tokens if not re.match(r'^[a-zA-Z\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc\u00f1\u00c1\u00c9\u00cd\u00d3\u00da\u00dc\u00d1]+$', t))
    oov = sum(1 for t in tokens if t.lower() not in modern_lexicon)
    corrupt = sum(1 for t in tokens if re.search(r'[^\\w\\s\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc\u00f1\u00c1\u00c9\u00cd\u00d3\u00da\u00dc\u00d1]', t))
    return {
        "ocr_symbol_rate": non_alpha * 100 / nw,
        "oov_rate": oov * 100 / nw,
        "corrupt_token_rate": corrupt * 100 / nw,
    }

ocr_feat_df = df_train["text_base"].apply(lambda t: pd.Series(extract_ocr_features(t)))
df_train = pd.concat([df_train, ocr_feat_df], axis=1)

print(f"OOV rate stats: mean={df_train['oov_rate'].mean():.2f}%, max={df_train['oov_rate'].max():.2f}%")
print(f"Corrupt token rate stats: mean={df_train['corrupt_token_rate'].mean():.2f}%")


### Paso 10. Extraer sintaxis y frecuencia léxica
**Objetivo:** capturar diferencias de época en estructura y vocabulario funcional.

**Output:** `historic_stopword_vector`, `citation_pattern_rate`, `ttr`, `subordination_rate`, `clause_len_mean`


In [ ]:
SUBORDINATORS = {"que", "pues", "donde", "si", "como", "cuando",
               "aunque", "porque", "mientras", "cual", "quien", "cuyo"}
STOPWORDS_HISTORIC = {"\u00e1", "\u00e9l", "ella", "ello", "ellos", "dixo",
                      "della", "dello", "dellas", "dellos", "deste", "desta",
                      "assi", "ans\u00ed", "ansi", "aunque", "pues", "ca", "mas",
                      "porque", "sobre", "entre", "cabe", "seg\u00fan", "sin", "con"}

def extract_syntactic_features(text):
    nw = max(len(text.split()), 1)
    tokens = text.lower().split()
    ttr = len(set(tokens)) / nw if tokens else 0
    subord_count = sum(1 for t in tokens if t in SUBORDINATORS)
    subordination_rate = subord_count * 100 / nw
    historic_stops = sum(1 for t in tokens if t in STOPWORDS_HISTORIC)
    historic_stopword_rate = historic_stops * 100 / nw
    clauses = re.split(r'[,.;:!?]+', text)
    clause_lens = [len(c.split()) for c in clauses if c.strip()]
    clause_len_mean = np.mean(clause_lens) if clause_lens else nw
    citation_pat = len(re.findall(
        r'[A-Z][a-z\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc\u00f1]+\\s+(?:lib|cap|pag|fol|epist)\\.\\s*\\d+', text))
    return {
        "ttr": round(ttr, 4),
        "subordination_rate": round(subordination_rate, 4),
        "historic_stopword_rate": round(historic_stopword_rate, 4),
        "clause_len_mean": round(clause_len_mean, 4),
        "citation_pattern_rate": citation_pat * 100 / nw,
    }

syn_df = df_train["text_base"].apply(lambda t: pd.Series(extract_syntactic_features(t)))
df_train = pd.concat([df_train, syn_df], axis=1)

print("Features sint\u00e1cticas extra\u00eddas.")
print(f"TTR mean: {df_train['ttr'].mean():.4f}, subordination mean: {df_train['subordination_rate'].mean():.4f}")


### Paso 11. Construir vistas de features para modelo híbrido
**Objetivo:** preparar matrices para entrenamiento multimodal: subword, char-level y features manuales.

**Output:** matrices listas para entrenamiento multimodal.


In [ ]:
FEATURE_COLUMNS = [
    "semicolon_rate", "emdash_rate", "colon_rate", "paren_rate",
    "quote_angle_rate", "excl_rate", "quest_rate", "comma_rate",
    "asterisk_rate", "dash_rate", "quote_double_rate", "sentence_len_mean",
    "obsolete_accent_rate", "long_s_proxy_rate", "latin_abbrev_rate",
    "honorific_caps_rate", "ocr_symbol_rate", "oov_rate", "corrupt_token_rate",
    "ttr", "subordination_rate", "historic_stopword_rate", "clause_len_mean",
    "citation_pattern_rate", "numeric_reference_density", "ocr_noise_score",
    "weird_char_rate", "alnum_mixed_token_rate", "count_linebreak_hyphen",
]

scaler = StandardScaler()
X_feats = scaler.fit_transform(df_train[FEATURE_COLUMNS].fillna(0))
df_feats = pd.DataFrame(X_feats, columns=[f"norm_{c}" for c in FEATURE_COLUMNS])
df_train = pd.concat([df_train, df_feats], axis=1)

import joblib
joblib.dump(scaler, RUTA_OUT / "feature_scaler.pkl")

print(f"Vista tabular: {X_feats.shape}")
print(f"Features normalizadas: {len(FEATURE_COLUMNS)}")
print(f"Scaler guardado")


---
## 3. Filtrado y eliminación


### Paso 12. Definir reglas mínimas de calidad
**Objetivo:** remover muestras que introducen más ruido que señal.

**Output:** subconjunto `train_valid` y tabla `discard_log` con motivo técnico.


In [ ]:
df_train["num_chars"] = df_train["text_base"].str.len()
df_train["num_words"] = df_train["text_base"].str.split().str.len()
df_train["alpha_ratio"] = df_train["text_base"].apply(
    lambda t: sum(1 for c in t if c.isalpha()) / max(len(t), 1))

UMBRAL_CHARS = 20
UMBRAL_WORDS = 3
UMBRAL_ALPHA = 0.3
UMBRAL_OCR_CRIT = 0.8

discard_log = []
keep_mask = pd.Series(True, index=df_train.index)

mask1 = df_train["num_chars"] < UMBRAL_CHARS
discard_log.extend([{"idx": i, "reason": "too_few_chars",
                     "value": df_train.loc[i, "num_chars"]} for i in df_train[mask1].index])
keep_mask &= ~mask1

mask2 = df_train["num_words"] < UMBRAL_WORDS
discard_log.extend([{"idx": i, "reason": "too_few_words",
                     "value": df_train.loc[i, "num_words"]} for i in df_train[mask2].index])
keep_mask &= ~mask2

mask3 = df_train["alpha_ratio"] < UMBRAL_ALPHA
discard_log.extend([{"idx": i, "reason": "low_alpha_ratio",
                     "value": round(df_train.loc[i, "alpha_ratio"], 4)} for i in df_train[mask3].index])
keep_mask &= ~mask3

mask4 = df_train["ocr_noise_score"] > UMBRAL_OCR_CRIT
discard_log.extend([{"idx": i, "reason": "critical_ocr_noise",
                     "value": round(df_train.loc[i, "ocr_noise_score"], 4)} for i in df_train[mask4].index])
keep_mask &= ~mask4

mask5 = df_train["numeric_reference_density"] > 0.5
discard_log.extend([{"idx": i, "reason": "excessive_references",
                     "value": round(df_train.loc[i, "numeric_reference_density"], 4)} for i in df_train[mask5].index])
keep_mask &= ~mask5

df_discard = pd.DataFrame(discard_log) if discard_log else pd.DataFrame()
print(f"Descartadas: {len(discard_log)} muestras")
if len(df_discard) > 0:
    print(f"Motivos:\n{df_discard['reason'].value_counts().to_string()}")


### Paso 13. Filtrado en dos niveles, no binario
**Objetivo:** no perder documentos históricamente valiosos por ruido moderado.

**Output:** dataset con `quality_tier = {high, medium, discard}`


In [ ]:
def assign_quality_tier(row):
    if row["ocr_noise_score"] > UMBRAL_OCR_CRIT or row["num_chars"] < UMBRAL_CHARS or row["alpha_ratio"] < UMBRAL_ALPHA:
        return "discard"
    elif row["ocr_noise_score"] > 0.3 or row["alpha_ratio"] < 0.5:
        return "medium"
    else:
        return "high"

df_train["quality_tier"] = df_train.apply(assign_quality_tier, axis=1)

print(f"Quality tiers:\n{df_train['quality_tier'].value_counts().to_string()}")
print(f"\nTasa de retención (high+medium): "
      f"{(df_train['quality_tier'].isin(['high','medium']).sum() / len(df_train)) * 100:.1f}%")


### Paso 14. Controlar leakage documental
**Objetivo:** evitar que fragmentos del mismo documento aparezcan en train y validación.

**Output:** folds limpios sin fuga de información.


In [ ]:
# Proxy de source_doc_id: hash del prefijo textual
df_train["source_doc_id"] = df_train["text_base"].str[:50].apply(hash) % 10000
df_train["source_doc_id"] = df_train["source_doc_id"].abs()

df_valid = df_train[df_train["quality_tier"] == "discard"]
df_keep = df_train[df_train["quality_tier"] != "discard"].copy()

gkk = GroupKFold(n_splits=5)
groups = df_keep["source_doc_id"]

for fold, (train_idx, val_idx) in enumerate(gkk.split(df_keep, df_keep["decade"], groups)):
    df_keep.loc[df_keep.iloc[train_idx].index, "fold"] = f"train_fold{fold}"
    df_keep.loc[df_keep.iloc[val_idx].index, "fold"] = f"val_fold{fold}"

for fold in range(5):
    train_docs = set(df_keep[df_keep["fold"] == f"train_fold{fold}"]["source_doc_id"])
    val_docs = set(df_keep[df_keep["fold"] == f"val_fold{fold}"]["source_doc_id"])
    overlap = train_docs & val_docs
    print(f"Fold {fold}: documentos solapados = {len(overlap)} "
          f"(train={len(train_docs)}, val={len(val_docs)})")

df_keep = df_keep.drop(columns=["fold"])
print(f"Dataset para entrenamiento: {len(df_keep)} muestras")
print(f"Dataset descartado: {len(df_valid)} muestras")


---
## 4. Aumentación y datos sintéticos


### Paso 15. Aumentación conservadora basada en ruido OCR simulado
**Objetivo:** volver el modelo robusto a escaneos y tipografías variables.

**Output:** `aug_text_ocr_sim` alineado con la misma etiqueta temporal.


In [ ]:
OCR_CONFUSIONS = {
    "s": ["f", "\u017f", "s"],
    "S": ["F", "S"],
    "u": ["v", "u"],
    "U": ["V", "U"],
    "v": ["u", "v"],
    "V": ["U", "V"],
    "i": ["j", "i"],
    "I": ["J", "I"],
    "j": ["i", "j"],
    "c": ["e", "c"],
    "C": ["E", "C"],
    "e": ["c", "e"],
    "E": ["C", "E"],
    "n": ["m", "n"],
    "m": ["n", "m"],
}

def augment_ocr_noise(text, prob=0.03):
    chars = list(text)
    for i in range(len(chars)):
        if np.random.random() < prob and chars[i] in OCR_CONFUSIONS:
            chars[i] = np.random.choice(OCR_CONFUSIONS[chars[i]])
    return "".join(chars)

df_medium = df_keep[df_keep["quality_tier"] == "medium"].copy()
if len(df_medium) > 0:
    df_medium["aug_text_ocr_sim"] = df_medium["text_base"].apply(
        lambda t: augment_ocr_noise(t, prob=0.05))
    print(f"Aumentadas {len(df_medium)} muestras con ruido OCR simulado")
else:
    df_medium["aug_text_ocr_sim"] = df_medium["text_base"]
    print("No hay muestras medium para aumentar")


### Paso 16. Aumentación por de/rehifenización de línea
**Objetivo:** entrenar al modelo a reconocer la misma muestra con y sin artefactos de composición.

**Output:** pares `{original, hyphenated_variant}`


In [ ]:
def simulate_hyphenation(text, prob=0.1):
    tokens = text.split()
    new_tokens = []
    for t in tokens:
        if len(t) >= 8 and np.random.random() < prob:
            split_point = len(t) // 2
            new_tokens.append(t[:split_point] + "-\\n" + t[split_point:])
        else:
            new_tokens.append(t)
    return " ".join(new_tokens)

df_high = df_keep[df_keep["quality_tier"] == "high"].copy()
df_high["hyphenated_variant"] = df_high["text_base"].apply(
    lambda t: simulate_hyphenation(t, prob=0.15))

df_high_aug = df_high.rename(columns={"hyphenated_variant": "text_base"})
df_high_aug["is_hyphenation_aug"] = 1
df_high["is_hyphenation_aug"] = 0

print(f"Muestras high originales: {len(df_high)}, variantes con hiphen: {len(df_high_aug)}")


### Paso 17. Enmascaramiento histórico controlado
**Objetivo:** evitar sobreajuste a tokens superficiales y reforzar señales globales de época.

**Output:** `aug_text_masked` que conserva la huella temporal.


In [ ]:
PROTECTED_TOKENS = {
    "lib.", "cap.", "pag.", "fol.", "epist.", "tom.", "vol.",
    "D.", "Don", "Do\u00f1a", "Exc.", "S.M.", "S.A.",
    "\u017f", "\u017fe", "\u017fu", "poreft", "della", "dello",
}

def historical_masking(text, mask_prob=0.08):
    tokens = text.split()
    masked = []
    for t in tokens:
        if t in PROTECTED_TOKENS:
            masked.append(t)
        elif np.random.random() < mask_prob and len(t) > 3:
            masked.append("[MASK]")
        else:
            masked.append(t)
    return " ".join(masked)

df_high["aug_text_masked"] = df_high["text_base"].apply(historical_masking)
masked_count = sum(1 for t in df_high["aug_text_masked"] if "[MASK]" in t)
print(f"Muestras con al menos un [MASK]: {masked_count} / {len(df_high)}")


### Paso 18. Aumentación léxica restringida, no libre
**Objetivo:** enriquecer sin modernizar el texto por accidente.

**Output:** variantes léxicas seguras y plausibles para la misma época.


In [ ]:
HISTORIC_SYNONYMS = {
    "dice": ["dize", "dice", "dixe"],
    "hacer": ["hazer", "hacer", "facer"],
    "mujer": ["muger", "mujer"],
    "decir": ["dezir", "decir", "dexir"],
    "as\u00ed": ["assi", "ans\u00ed", "ansi", "as\u00ed"],
    "mismo": ["mismo", "mesmo"],
    "escribir": ["escrevir", "escribir"],
    "cosa": ["cosa", "cossa"],
    "a\u00f1o": ["a\u00f1o", "anno"],
    "tiempo": ["tiempo", "tienpo", "tyempo"],
    "autor": ["autor", "auth\u00f3r", "auctor"],
    "parecer": ["parecer", "parescer"],
    "raz\u00f3n": ["raz\u00f3n", "ra\u00e7\u00f3n", "razon"],
    "hombre": ["hombre", "ombre"],
}

def restricted_lexical_augment(text, prob=0.1):
    tokens = text.split()
    augmented = []
    changed = False
    for t in tokens:
        t_lower = t.lower()
        if t_lower in HISTORIC_SYNONYMS and np.random.random() < prob:
            options = HISTORIC_SYNONYMS[t_lower]
            chosen = np.random.choice(options)
            if t[0].isupper():
                chosen = chosen.capitalize()
            augmented.append(chosen)
            changed = True
        else:
            augmented.append(t)
    return " ".join(augmented), changed

results_lex = df_high["text_base"].apply(restricted_lexical_augment)
df_high["aug_text_lexical"] = [r[0] for r in results_lex]
lex_changed = sum(1 for r in results_lex if r[1])
print(f"Muestras con sustituci\u00f3n l\u00e9xica: {lex_changed} / {len(df_high)}")


### Paso 19. Balanceo de clases por oversampling de textos, no SMOTE textual
**Objetivo:** mejorar representación de décadas minoritarias sin crear vectores irreales.

**Output:** distribución de décadas más equilibrada.


In [ ]:
def oversample_minority(df, target_col="decade", augment_col="text_base"):
    counts = df[target_col].value_counts()
    max_count = counts.max()
    augmented_rows = []
    for cls, count in counts.items():
        if count < max_count:
            cls_df = df[df[target_col] == cls]
            n_needed = max_count - count
            n_copies = n_needed // count
            remainder = n_needed % count
            for _ in range(n_copies):
                aug = cls_df.copy()
                aug["is_oversampled"] = 1
                augmented_rows.append(aug)
            if remainder > 0:
                aug = cls_df.sample(n=remainder, random_state=42).copy()
                aug["is_oversampled"] = 1
                augmented_rows.append(aug)
    if augmented_rows:
        df_over = pd.concat([df] + augmented_rows, ignore_index=True)
    else:
        df_over = df.copy()
    df_over["is_oversampled"] = df_over.get("is_oversampled", 0)
    return df_over

df_high["is_oversampled"] = 0
df_balanced = oversample_minority(df_high, target_col="decade")

print(f"Balanceo: {len(df_high)} -> {len(df_balanced)} muestras")
print(f"Distribuci\u00f3n post-balanceo:\n{df_balanced['decade'].value_counts().sort_index()}")


### Paso 20. Política de mezcla de originales y sintéticos
**Objetivo:** impedir que el modelo aprenda artefactos artificiales en exceso.

**Output:** conjunto final balanceado con diversidad controlada.


In [ ]:
final_rows = []

# 60-80% originales
n_original = int(len(df_keep) * 0.7)
df_sample_orig = df_keep.sample(n=n_original, random_state=42)
df_sample_orig["aug_type"] = "original"
final_rows.append(df_sample_orig)

n_aug = len(df_keep) - n_original

if len(df_medium) > 0:
    aug_ocr = df_medium.sample(n=min(n_aug // 3, len(df_medium)), random_state=42).copy()
    aug_ocr["aug_type"] = "ocr_simulated"
    aug_ocr["text_base"] = aug_ocr["aug_text_ocr_sim"]
    final_rows.append(aug_ocr)

aug_hyphen = df_high_aug.sample(n=min(n_aug // 3, len(df_high_aug)), random_state=42).copy()
aug_hyphen["aug_type"] = "hyphenation"
final_rows.append(aug_hyphen)

aug_masked = df_high.sample(n=min(n_aug // 3, len(df_high)), random_state=42).copy()
aug_masked["aug_type"] = "masked"
aug_masked["text_base"] = aug_masked["aug_text_masked"]
final_rows.append(aug_masked)

df_final = pd.concat(final_rows, ignore_index=True)
print(f"Dataset final: {len(df_final)} muestras")
print(f"Composici\u00f3n:\n{df_final['aug_type'].value_counts()}")
print(f"\nDistribuci\u00f3n por d\u00e9cada:\n{df_final['decade'].value_counts().sort_index()}")


---
## 5. Secuencia final ejecutable


### Paso 21. Pipeline completo
**Objetivo:** pipeline completo ejecutable de principio a fin.

**Output:** `train.parquet`, `valid.parquet`, `test.parquet`, `feature_schema.json`, `quality_report.csv`


In [ ]:
# === PIPELINE COMPLETO ===

def run_full_pipeline(df_input):
    """Ejecuta pasos 1-20 en secuencia."""
    df = df_input.copy()

    # Paso 1
    utf8_result = validate_utf8_series(df["text"])
    df = pd.concat([utf8_result, df.drop(columns=["text"])], axis=1)

    # Paso 2
    df["text_raw"] = df["raw_text"]
    df["text_base"] = df["text_utf8"]
    df["text_norm_soft"] = df["text_utf8"].apply(normalize_whitespace_soft)

    # Paso 3
    df["text_base"] = df["text_base"].apply(strip_control_chars)
    df["text_norm_soft"] = df["text_norm_soft"].apply(strip_control_chars)

    # Paso 4
    hyphen_results = df["text_base"].apply(resolve_linebreak_hyphens)
    df["text_dehyphenated"] = [r[0] for r in hyphen_results]
    df["count_linebreak_hyphen"] = [r[1] for r in hyphen_results]
    df["had_linebreak_hyphen"] = (df["count_linebreak_hyphen"] > 0).astype(int)

    # Paso 5
    ed_results = df["text_base"].apply(detect_editorial_noise)
    df["citation_flag"] = [r["citation_flag"] for r in ed_results]
    df["numeric_reference_density"] = [r["numeric_reference_density"] for r in ed_results]

    # Paso 6
    ocr_results = df["text_base"].apply(score_ocr_noise)
    df["ocr_noise_score"] = [r["ocr_noise_score"] for r in ocr_results]
    df["weird_char_rate"] = [r["weird_char_rate"] for r in ocr_results]

    # Pasos 7-10: Features
    stylo = df["text_base"].apply(lambda t: pd.Series(extract_stylometry(t)))
    hist = df["text_base"].apply(lambda t: pd.Series(extract_historical_spelling(t)))
    ocr_f = df["text_base"].apply(lambda t: pd.Series(extract_ocr_features(t)))
    syn = df["text_base"].apply(lambda t: pd.Series(extract_syntactic_features(t)))
    df = pd.concat([df, stylo, hist, ocr_f, syn], axis=1)

    # Paso 11: Normalize
    scaler = StandardScaler()
    X_feats = scaler.fit_transform(df[FEATURE_COLUMNS].fillna(0))
    for i, c in enumerate(FEATURE_COLUMNS):
        df[f"norm_{c}"] = X_feats[:, i]

    # Pasos 12-13: Quality
    df["num_chars"] = df["text_base"].str.len()
    df["num_words"] = df["text_base"].str.split().str.len()
    df["alpha_ratio"] = df["text_base"].apply(lambda t: sum(1 for c in t if c.isalpha()) / max(len(t), 1))
    df["quality_tier"] = df.apply(assign_quality_tier, axis=1)

    return df, scaler

print("Pipeline function defined.")


In [ ]:
# Ejecutar pipeline y guardar outputs
df_fresh = pd.read_csv(RUTA_DATA / "train.csv")
df_processed, scaler = run_full_pipeline(df_fresh)

df_train_out = df_processed[df_processed["quality_tier"] != "discard"]
df_valid_out = df_processed[df_processed["quality_tier"] == "discard"]

text_cols = ["text_raw", "text_base", "text_norm_soft", "text_dehyphenated"]
feat_cols = FEATURE_COLUMNS
meta_cols = ["decade", "quality_tier", "ocr_noise_score", "num_chars", "num_words"]
keep_cols = [c for c in text_cols + feat_cols + meta_cols if c in df_train_out.columns]

df_train_out[keep_cols].to_parquet(RUTA_OUT / "train.parquet", index=False)
df_valid_out[keep_cols].to_parquet(RUTA_OUT / "valid.parquet", index=False)

# Procesar eval - preservar columna id original, NO renombrar
df_eval_proc = df_eval.copy()
if "decade" not in df_eval_proc.columns:
    df_eval_proc["decade"] = -1  # dummy para que el pipeline no falle
df_eval_proc, _ = run_full_pipeline(df_eval_proc)
# Guardar manteniendo id original para submission
keep_cols_eval = [c for c in keep_cols if c != "decade"]
df_eval_out = df_eval[["id"]].copy()
for c in keep_cols_eval:
    if c in df_eval_proc.columns:
        df_eval_out[c] = df_eval_proc[c].values
df_eval_out.to_parquet(RUTA_OUT / "test.parquet", index=False)

# Schema
with open(RUTA_OUT / "feature_schema.json", "w") as f:
    json.dump({"feature_columns": FEATURE_COLUMNS}, f)

df_processed["quality_tier"].value_counts().to_csv(RUTA_OUT / "quality_report.csv")

print("=== OUTPUTS GENERADOS ===")
print(f"train.parquet: {len(df_train_out)} filas")
print(f"valid.parquet: {len(df_valid_out)} filas")
print(f"test.parquet: {len(df_eval_proc)} filas")
print(f"feature_schema.json: {len(FEATURE_COLUMNS)} features")


### Paso 22. Validación de impacto
**Objetivo:** comprobar que cada fase realmente mejora accuracy y no destruye señal temporal.

**Output:** tabla de ganancia incremental por módulo.

> **⚠️ Advertencia:** La ablación usa RandomForest sobre features tabulares. Los números reflejan únicamente la utilidad relativa de las features para un modelo clásico, NO predicen el accuracy final de un transformer como DeBERTa/BERT. Usa esto para **seleccionar/priorizar features**, no para decidir arquitectura.


In [ ]:
def evaluate_ablation(df, config_name, feature_cols):
    """Evalúa features con RandomForest. NO refleja accuracy de transformer."""
    X = df[feature_cols].fillna(0).values
    y = df["decade"].values
    X_tr, X_va, y_tr, y_va = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)
    clf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_va)
    return {
        "config": config_name,
        "accuracy": round(accuracy_score(y_va, y_pred), 4),
        "macro_f1": round(f1_score(y_va, y_pred, average="macro"), 4),
        "n_features": X.shape[1],
    }

ablations = []

base_feats = ["num_chars", "num_words"]
stylo_feats = [c for c in FEATURE_COLUMNS if c in [
    "semicolon_rate", "emdash_rate", "colon_rate", "comma_rate", "sentence_len_mean"]]
hist_feats = ["obsolete_accent_rate", "long_s_proxy_rate",
              "latin_abbrev_rate", "honorific_caps_rate"]
ocr_feats = ["ocr_noise_score", "ocr_symbol_rate", "oov_rate", "corrupt_token_rate"]
syn_feats = ["ttr", "subordination_rate", "historic_stopword_rate",
             "clause_len_mean", "citation_pattern_rate"]

ablations.append(evaluate_ablation(df_processed, "baseline", base_feats))
ablations.append(evaluate_ablation(df_processed, "+stylometry", base_feats + stylo_feats))
ablations.append(evaluate_ablation(df_processed, "+historical_spelling", base_feats + hist_feats))
ablations.append(evaluate_ablation(df_processed, "+ocr_features", base_feats + ocr_feats))
ablations.append(evaluate_ablation(df_processed, "+syntax", base_feats + syn_feats))
ablations.append(evaluate_ablation(df_processed, "all_features", FEATURE_COLUMNS))

df_ablation = pd.DataFrame(ablations)
print("\n=== ABLATION STUDY ===")
print(df_ablation.to_string(index=False))
print("\n--- Ganancia incremental ---")
for i in range(1, len(df_ablation)):
    prev = df_ablation.iloc[i-1]
    curr = df_ablation.iloc[i]
    print(f"{curr['config']}: acc +{curr['accuracy'] - prev['accuracy']:.4f}, "
          f"f1 +{curr['macro_f1'] - prev['macro_f1']:.4f}")


---
## Resumen final del pipeline

| Componente | Acción clave | Impacto esperado |
|---|---|---|
| UTF-8 validation | Codificación consistente | Evita errores de decoder |
| Dual text view | text_base + text_norm_soft | Preserva + robustece |
| Dehyphenation | Reconstruye palabras partidas | +1-2% accuracy s. XVIII-XIX |
| Editorial tagging | Marca citas/referencias | Reduce falsos positivos |
| OCR scoring | Etiqueta corrupción | Pesos diferenciales en training |
| Stylometry (23 feats) | Puntuación, longitud, TTR | +5-8% sobre baseline |
| Historical spelling | \u017f, tildes, abreviaturas | +3-5% en s. XVI-XVII |
| Quality filtering | 2-tier + discard | Elimina ~5% ruido extremo |
| Grouped split | Por documento sin leakage | Validez estadística |
| OCR augmentation | Confusiones realistas | +1-3% robustez |
| Hyphenation aug | Variantes de corte | +1% consistencia |
| Historical masking | Protege señal temporal | Generalización |
| Lexical augmentation | Sinónimos históricos | +1-2% clases minoritarias |
| Oversampling | Balanceo sin SMOTE textual | +2-4% macro F1 |

**Regla de oro:** no limpiar todo — en textos históricos, parte del "ruido" es la señal temporal.
